## This notebook provides the template to run post hoc explainers including GNNExplainer, PGExplainer on the optc dataset.

## You can choose to download the data yourself using the urls mentioned in this notebook or mount this notebook to the data directory in Beryl to run it automatically. The directory information is present in the repository Readme.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import torch
from torch_geometric.data import Data
import os
import torch.nn.functional as F
import json
import warnings
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
warnings.filterwarnings('ignore')
from torch_geometric.loader import NeighborLoader

import os
import json
import pickle
import time

import torch
import numpy as np
from torch.nn import CrossEntropyLoss
from torch_geometric.data import Data
from torch_geometric.loader import NeighborLoader
from torch_geometric import utils
from sklearn.utils import class_weight

from pprint import pprint
import gzip
from sklearn.manifold import TSNE
import json
import copy
import os

import gensim
from gensim.models import Word2Vec
from multiprocessing import Pool
from itertools import compress
from tqdm import tqdm
import time

import math
import multiprocessing

import gensim
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
from collections import Counter
from gensim.models import Word2Vec
from multiprocessing import Pool
from itertools import compress
from tqdm import tqdm
import time

from torch_geometric.nn import GCNConv
from torch_geometric.nn import SAGEConv, GATConv
import torch.nn.functional as F
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
%matplotlib inline

In [ ]:
os.chdir("content")
%pwd

In [ ]:
class PositionalEncoding():

    def __init__(self, d_model ,max_len = 100000):
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        self.pe = torch.zeros(max_len, d_model)
        self.pe[:,0::2] = torch.sin(position * div_term)
        self.pe[:,1::2] = torch.cos(position * div_term)

    def embed(self, x):
        x = x + self.pe[:x.size(0)]
        return x

encoder = PositionalEncoding(20)

In [ ]:
def preprocess(data):
  new_data = {}
  for x in data:
    check1 = x['object'] in ['PROCESS','FILE','FLOW','MODULE']
    check2 = not (x['action'] in ['START','TERMINATE'])
    check3 = x['actorID'] != x['objectID']
    key = (x['action'],x['actorID'],x['objectID'],x['object'],x['pid'],x['ppid'])
    if check1 and check2 and check3:
      new_data[key] = x
  return list(new_data.values())

In [ ]:
def describe(x):
  action = x["action"]
  props = x['properties']
  typ = x['object']

  phrase = ''
  try:
    if typ == 'PROCESS':
        phrase = f"{props['parent_image_path']} {action} {props['image_path']} {props['command_line']}"    

    elif typ == 'FILE':
        phrase = f"{props['image_path']} {action} {props['file_path']}"    

    elif typ == 'FLOW':
        phrase = f"{props['image_path']} {action} {props['src_ip']} {props['src_port']} {props['dest_ip']} {props['dest_port']} {props['direction']}"    

    else:
        phrase = f"{props['image_path']} {action} {props['module_path']}"
  except:
      phrase = ''
  
  return phrase.split(' ')

In [ ]:
'''
The following function loads the data in streaming mode. It loads n number of rows from the dataset at a time.
It then creates a graphraph from these n rows and extract the features of the nodes in the graph.
We can define n using the batch parameter in the following function.
'''
def transform(text):
    lbldta = [get_labels(x) for x in text]
    lbldta = [x for x in lbldta if x != None]

    data = preprocess(lbldta)

    temp = [describe(x) for x in data]
    temp = [x for x in temp if len(x) != 0]

    for i in range(len(data)):
        data[i]['phrase'] = temp[i]

    df = pd.DataFrame.from_dict(data)
    df['timestamp'] = df['timestamp'].str[:-6]
    df['timestamp'] = pd.to_datetime(df['timestamp'],infer_datetime_format=True)
    df.sort_values(by='timestamp', ascending=True,inplace=True)

    return df

def get_labels(x):
  event = x
  typ = event['object']
  props = event['properties']
  try:
    if typ == "PROCESS":
      event["actorname"] = props['parent_image_path']
      event["objectname"] = props['image_path']

    if typ == "FILE":
      event["actorname"] = props['image_path'] 
      event["objectname"] = props['file_path']
    
    if typ == "MODULE":
      event["actorname"] = props['image_path']
      event["objectname"] = props['module_path']
    
    if typ == "FLOW":
      event["actorname"] = props['image_path']
      event["objectname"] = props['dest_ip']+' '+props['dest_port']
    
    return event
  except:
    return None

def load_data(name=None):
    if name == None:
        rawdata = []
        for i in range(1,2):
            f = open(f"singleHost/201_{i}.txt")
            content = [json.loads(line) for line in f]
            rawdata = rawdata + content
        return prepare_graph(transform(rawdata))
    else:
        f = open(name)
        content = [json.loads(line) for line in f]
        return prepare_graph(transform(content))

In [ ]:
import time
def prepare_graph(df):
  nodes = {}
  labels = {}
  edges = []
    
  dummies = {'PROCESS':0,'FLOW':1,'FILE':2,'MODULE':3}

  lblmap = {}
  neimap = {}

  for i in range(len(df)):
    x = df.iloc[i]

    actorid = x['actorID']
    objectid = x["objectID"]

    if not (actorid in nodes):
      nodes[actorid] = []
    if not (objectid in nodes):
      nodes[objectid] = []
    
    nodes[actorid] += x['phrase']
    nodes[objectid] += x['phrase']

    labels[actorid] = dummies['PROCESS']
    labels[objectid] = dummies[x['object']]
    
    lblmap[actorid] = x['actorname']
    lblmap[objectid] = x['objectname']
    
    if not (actorid in neimap):
      neimap[actorid] = set()
    if not (objectid in neimap):
      neimap[objectid] = set()
    
    neimap[actorid].add(x['objectname'])
    neimap[objectid].add(x['actorname'])
    
    if x['object'] == 'FLOW':
        edges.append(( actorid, objectid, x['properties']['direction'] ))
    else:
        edges.append(( actorid, objectid, x['action'] ))

  features = []
  feat_labels = []
  edge_index = [[],[]]
  index  = {}
  mapp = []
              
  for k,v in nodes.items():
    if not ((len(v)==1) and (v[0] == 'DELETE')):
        features.append(v)
        feat_labels.append(labels[k])
        index[k] = len(features) - 1
        mapp.append(k)

  for x in edges:
    src = index[x[0]]
    dst = index[x[1]]
    
    edge_index[0].append(src)
    edge_index[1].append(dst)    
    
  return features,feat_labels,edge_index,mapp,lblmap, neimap

In [ ]:
class GCN(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = SAGEConv(20, 32, normalize=True)
        self.conv2 = SAGEConv(32, 4, normalize=True)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = x.relu()
        x = F.dropout(x, p=0.5, training=self.training)

        x = self.conv2(x, edge_index)
        return F.softmax(x, dim=1)

In [ ]:
from gensim.models.callbacks import CallbackAny2Vec

class EpochSaver(CallbackAny2Vec):
    '''Callback to save model after each epoch.'''

    def __init__(self):
        self.epoch = 0

    def on_epoch_end(self, model):
        model.save('word2vec_optc.model')
        self.epoch += 1

In [ ]:
class EpochLogger(CallbackAny2Vec):
    '''Callback to log information about training'''

    def __init__(self):
        self.epoch = 0

    def on_epoch_begin(self, model):
        print("Epoch #{} start".format(self.epoch))

    def on_epoch_end(self, model):
        print("Epoch #{} end".format(self.epoch))
        self.epoch += 1

In [ ]:
logger = EpochLogger()
saver = EpochSaver()

word2vec = Word2Vec.load("word2vec2.model")

def infer(doc):  
  word_emb = []
  for word in doc:
    if word in word2vec.wv:
      word_emb.append(word2vec.wv[word])
  
  if len(word_emb) == 0:
    return np.zeros(30)

  out_emb = torch.tensor(word_emb,dtype=torch.float)
  if len(doc) < 100000:
    out_emb = encoder.embed(out_emb)
  out_emb = out_emb.detach().cpu().numpy()
  out_emb = np.mean(out_emb,axis=0)
  return out_emb

In [ ]:
import torch.nn.functional as F
from torch.nn import CrossEntropyLoss

In [ ]:
from itertools import compress
from torch_geometric import utils

def construct_neighborhood(ids,mapp,edges,hops):
    if hops == 0:
        return set()
    else:
        neighbors = set()
        for i in range(len(edges[0])):
            if mapp[edges[0][i]] in ids:
                neighbors.add(mapp[edges[1][i]])
            if mapp[edges[1][i]] in ids:
                neighbors.add(mapp[edges[0][i]])
        return neighbors.union( construct_neighborhood(neighbors,mapp,edges,hops-1) )

In [ ]:
def helper(MP,all_pids,GP,edges,mapp):
    
    TP = MP.intersection(GP)  
    FP = MP - GP              
    FN = GP - MP              
    TN = all_pids - (GP | MP)
    
    two_hop_gp = construct_neighborhood(GP,mapp,edges,0)
    two_hop_tp = construct_neighborhood(TP,mapp,edges,0)
    FPL = FP - two_hop_gp
    TPL = TP.union(FN.intersection(two_hop_tp))
    FN = FN - two_hop_tp

    TP,FP,FN,TN = len(TPL),len(FPL),len(FN),len(TN)
    
    FPR = FP / (FP+TN)
    TPR = TP / (TP+FN)

    print(f"Number of True Positives: {TP}")
    print(f"Number of Fasle Positives: {FP}")
    print(f"Number of False Negatives: {FN}")
    print(f"Number of True Negatives: {TN}\n")

    prec = TP / (TP + FP)
    print(f"Precision: {round(prec, 2)}")

    rec = TP / (TP + FN)
    print(f"Recall: {round(rec, 2)}")

    fscore = (2*prec*rec) / (prec + rec)
    print(f"Fscore: {round(fscore, 2)}\n")
    
    return TPL,FPL

In [ ]:
f = open(f"singleHost/501_1.txt")
content = [json.loads(line) for line in f]
df_ben = transform(content)

In [ ]:
model = GCN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

phrases, labels, edges, mapp, lbl, nemap = prepare_graph(df_ben)

nodes = [infer(x) for x in phrases]
nodes = np.array(nodes)

criterion = CrossEntropyLoss()

graph = Data(
    x=torch.tensor(nodes, dtype=torch.float).to(device),
    y=torch.tensor(labels, dtype=torch.long).to(device),
    edge_index=torch.tensor(edges, dtype=torch.long).to(device)
)

for epochs in range(100):
    model.train()
    optimizer.zero_grad() 
    out = model(graph.x, graph.edge_index) 
    loss = criterion(out, graph.y) 
    loss.backward() 
    optimizer.step()      
    total_loss = loss.item()  
    print(total_loss)

torch.save(model.state_dict(), f"lword2vec_gnn_optc_explain.pth")

In [ ]:
f = open(f"../script_test/all_system_events")
content = [json.loads(line) for line in f]
df_mal = transform(content)

phrases, labels, edges, mapp, lbl, nemap = prepare_graph(df_mal)
nodes = [infer(x) for x in phrases]
nodes = np.array(nodes)

graph = Data(
    x=torch.tensor(nodes, dtype=torch.float).to(device),
    y=torch.tensor(labels, dtype=torch.long).to(device),
    edge_index=torch.tensor(edges, dtype=torch.long).to(device)
)

model = GCN().to(device)
model.load_state_dict(torch.load(f"lword2vec_gnn_optc_explain.pth",))
model.eval() 

In [ ]:
print(len(df_mal))

In [ ]:
node_labels = [lbl[x] for x in mapp]
graph.node_labels = node_labels

In [ ]:
indices = []
for i in range(len(labels)):
    if labels[i] == 0:
        indices.append(i)

In [ ]:
indices = []
for i in range(len(labels)):
    indices.append(i)

In [ ]:
for i in range(len(indices)):
    node_idx = indices[i]
    if "runme" in lbl[mapp[node_idx]]:
        print(i, lbl[mapp[node_idx]])

In [ ]:
from torch_geometric.explain import Explainer, GNNExplainer, ModelConfig
import torch

node_idx = indices[132023]
print(lbl[mapp[node_idx]])

explainer = Explainer(
    model=model,
    algorithm=GNNExplainer(epochs=200),
    explanation_type='model',
    node_mask_type='attributes',
    edge_mask_type='object',
    model_config=dict(
        mode='multiclass_classification',
        task_level='node',
        return_type='log_probs',
    ),
)

explanation = explainer(
    x=graph.x,
    edge_index=graph.edge_index,
    target=graph.y,
    index=node_idx,
)

In [ ]:
from pyvis.network import Network
import networkx as nx
from torch_geometric.utils import k_hop_subgraph

# Extract k-hop subgraph and compute normalized edge_mask
subset, edge_index, mapping, hard_edge_mask = k_hop_subgraph(
    node_idx, num_hops=2, edge_index=graph.edge_index, relabel_nodes=True
)
edge_mask = explanation.edge_mask[hard_edge_mask].cpu().detach().numpy()
edge_mask = (edge_mask - edge_mask.min()) / (edge_mask.max() - edge_mask.min())

# Build a NetworkX graph from the subgraph edges
G = nx.Graph()
edge_list = edge_index.cpu().numpy().T
G.add_edges_from(edge_list)

# Retrieve the node labels for the subgraph
subset_indices = subset.cpu().numpy()
node_labels = [graph.node_labels[i] for i in subset_indices]

# Create a PyVis Network
net = Network(notebook=True, cdn_resources='in_line', width='100%', height='750px')

# Add nodes to the visualization
for i, node in enumerate(G.nodes()):
    node_int = int(node) 
    label = node_labels[i]
    short_label = label if len(label) <= 15 else label.split('\\')[-1]  # shorten long labels
    title = f"Node {subset_indices[i]}: {label}"
    color = 'red' if int(subset_indices[i]) == int(node_idx) else 'blue'
    net.add_node(node_int, label=short_label, title=title, color=color)

# Highlight edges above the threshold and dim others
threshold = 0.2
for i, (u, v) in enumerate(G.edges()):
    u_int, v_int = int(u), int(v)
    weight = float(edge_mask[i])
    
    # Choose edge color based on threshold
    color = 'red' if weight >= threshold else 'grey'
    
    # You can also make edges thinner/thicker based on importance by tweaking 'value' or adding 'width'
    net.add_edge(
        u_int, 
        v_int, 
        #value=weight, 
        title=f"Edge weight: {weight:.2f}", 
        color=color
    )

# Optional: Configure physics layout
net.set_options("""
var options = {
  "physics": {
    "forceAtlas2Based": {
      "gravitationalConstant": -50,
      "centralGravity": 0.01,
      "springLength": 100,
      "springConstant": 0.08
    },
    "maxVelocity": 50,
    "solver": "forceAtlas2Based",
    "timestep": 0.35,
    "stabilization": {"iterations": 150}
  }
}
""")

# Show the resulting visualization
net.show('../graph_explanation.html')


In [ ]:
from torch_geometric.explain import Explainer, PGExplainer, ModelConfig, GNNExplainer
from torch_geometric.utils import k_hop_subgraph
import torch

all_explainer_edges_pg = []
for node_idx in node_indices:
    
    explainer = Explainer(
        model=model,
        algorithm=PGExplainer(epochs=30, lr=0.003),
        explanation_type='phenomenon',
        edge_mask_type='object',
        model_config=ModelConfig(
            mode='multiclass_classification',
            task_level='node',
            return_type='log_probs'
        ),
    )
    
    # Move the PGExplainer's MLP to the CUDA device
    explainer.algorithm.mlp.to(device)
    
    for epoch in range(30):
        for index in [node_idx]: 
            loss = explainer.algorithm.train(
                epoch,
                model.to(device),
                graph.x.to(device),
                graph.edge_index.to(device),
                target=graph.y.to(device),
                index=index
            )
    
    explanation = explainer(
        x=graph.x.to(device),
        edge_index=graph.edge_index.to(device),
        target=graph.y.to(device),
        index=node_idx,
    )
    
    explainer_edges = explanation.edge_mask
    indices = (explainer_edges > 0).nonzero(as_tuple=True)[0].tolist()
    indices = set(indices)
    all_explainer_edges_pg.extend(indices)

    #subset, edge_index, mapping, hard_edge_mask = k_hop_subgraph(
    #    node_idx, num_hops=2, edge_index=graph.edge_index, relabel_nodes=True
    #)
    #subset_indices = subset.cpu().numpy()
    #node_labels = [graph.node_ids[i] for i in subset_indices]

gt_edge_indices = set()
for i in range(graph.edge_index.shape[1]):
    src, dst = graph.edge_index[0][i], graph.edge_index[1][i]
    if graph.node_ids[src] in gt_coverage and graph.node_ids[dst] in gt_coverage: 
        gt_edge_indices.add(i) 

all_explainer_edges = set(all_explainer_edges_pg)
gt_edge_indices = set(gt_edge_indices)  

tp = len(all_explainer_edges.intersection(gt_edge_indices))
fp = len(all_explainer_edges - gt_edge_indices)
fn = len(gt_edge_indices - all_explainer_edges)

N = len(graph.edge_index[0])
total_edges = set(range(N))  
tn = len(total_edges - all_explainer_edges - gt_edge_indices)

# Print metrics
print(f'True Positives (TP): {tp}')
print(f'False Positives (FP): {fp}')
print(f'False Negatives (FN): {fn}')
print(f'True Negatives (TN): {tn} \n')